# 03 — Model Training & Evaluation

Train and evaluate classifiers on two feature sets x two split strategies:
- **Feature sets**: Correlation-filtered vs PCA-transformed
- **Split strategies**: Random (stratified 80/20) vs Temporal (train pre-2020, test 2020–2021)

Models: Random Forest, Gradient Boosting, SVM, and XGBoost.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from time import time

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score,
    ConfusionMatrixDisplay, roc_curve, auc
)
from sklearn.preprocessing import label_binarize

sns.set_style("whitegrid")

## 1. Load Processed Data

In [ ]:
def load_split(split_name):
    """Load a processed split (random or temporal) from disk."""
    d = f"../data/processed/{split_name}"
    return {
        "X_train_filt": pd.read_csv(f"{d}/X_train_filtered.csv", index_col=0),
        "X_test_filt": pd.read_csv(f"{d}/X_test_filtered.csv", index_col=0),
        "X_train_pca": pd.read_csv(f"{d}/X_train_pca.csv", index_col=0),
        "X_test_pca": pd.read_csv(f"{d}/X_test_pca.csv", index_col=0),
        "y_train": pd.read_csv(f"{d}/y_train.csv", index_col=0).squeeze(),
        "y_test": pd.read_csv(f"{d}/y_test.csv", index_col=0).squeeze(),
        "le": joblib.load(f"{d}/label_encoder.pkl"),
    }

random_data = load_split("random")
temporal_data = load_split("temporal")

for name, data in [("Random", random_data), ("Temporal", temporal_data)]:
    print(f"\n{name} split:")
    print(f"  Filtered: train={data['X_train_filt'].shape}, test={data['X_test_filt'].shape}")
    print(f"  PCA:      train={data['X_train_pca'].shape}, test={data['X_test_pca'].shape}")
    print(f"  Classes:  {list(data['le'].classes_)}")

## 2. Define Models and Training Helper

In [ ]:
def make_models():
    """Create fresh model instances."""
    m = {
        "Random Forest": RandomForestClassifier(
            n_estimators=200, random_state=42, n_jobs=-1
        ),
        "Gradient Boosting": GradientBoostingClassifier(
            n_estimators=200, max_depth=5, learning_rate=0.1, random_state=42
        ),
        "SVM (RBF)": SVC(
            kernel="rbf", C=10, gamma="scale", random_state=42, probability=True
        ),
    }
    try:
        from xgboost import XGBClassifier
        m["XGBoost"] = XGBClassifier(
            n_estimators=200, max_depth=6, learning_rate=0.1,
            random_state=42, n_jobs=-1, eval_metric="mlogloss"
        )
    except ImportError:
        print("XGBoost not installed — skipping.")
    return m


def train_and_evaluate(models, X_train, X_test, y_train, y_test, label):
    """Train each model, return results dict."""
    results = {}
    for name, model in models.items():
        print(f"  Training {name}...", end=" ")
        t0 = time()
        model.fit(X_train, y_train)
        train_time = time() - t0

        y_pred = model.predict(X_test)
        acc = accuracy_score(y_test, y_pred)
        print(f"accuracy={acc:.4f}  ({train_time:.1f}s)")

        results[name] = {
            "model": model, "y_pred": y_pred, "accuracy": acc,
            "train_time": train_time, "label": label,
        }
    return results

print(f"Models: {list(make_models().keys())}")

## 3. Train All Combinations

Train each model on 4 combinations: {Random, Temporal} x {Filtered, PCA}.

In [ ]:
all_results = {}

for split_name, data in [("Random", random_data), ("Temporal", temporal_data)]:
    for feat_name, feat_key in [("Filtered", "filt"), ("PCA", "pca")]:
        combo = f"{split_name} / {feat_name}"
        print(f"\n{'='*50}")
        print(f"{combo} (train={data[f'X_train_{feat_key}'].shape})")
        print(f"{'='*50}")

        results = train_and_evaluate(
            make_models(),
            data[f"X_train_{feat_key}"], data[f"X_test_{feat_key}"],
            data["y_train"], data["y_test"],
            label=combo,
        )

        for model_name, res in results.items():
            res["split"] = split_name
            res["feature_set"] = feat_name
            res["le"] = data["le"]
            all_results[f"{model_name} ({combo})"] = res

print(f"\nTotal experiments: {len(all_results)}")

## 4. Classification Reports

In [ ]:
for name, res in all_results.items():
    split_data = random_data if res["split"] == "Random" else temporal_data
    print(f"\n{'='*60}")
    print(f"{name}  —  Accuracy: {res['accuracy']:.4f}")
    print(f"{'='*60}")
    print(classification_report(
        split_data["y_test"], res["y_pred"], target_names=res["le"].classes_
    ))

## 5. Accuracy Comparison — Random vs Temporal Split

In [ ]:
summary = pd.DataFrame([
    {"Experiment": name, "Model": name.split(" (")[0],
     "Split": res["split"], "Feature Set": res["feature_set"],
     "Accuracy": res["accuracy"], "Train Time (s)": round(res["train_time"], 1)}
    for name, res in all_results.items()
]).sort_values("Accuracy", ascending=False).reset_index(drop=True)

summary

In [ ]:
# Side-by-side comparison: Random vs Temporal for each model+feature combo
from matplotlib.patches import Patch

pivot = summary.pivot_table(index=["Model", "Feature Set"], columns="Split", values="Accuracy")
pivot = pivot.sort_values("Random", ascending=True)

fig, ax = plt.subplots(figsize=(12, 8))
y_pos = range(len(pivot))
bar_height = 0.35

bars1 = ax.barh([y - bar_height/2 for y in y_pos], pivot["Random"], bar_height,
                label="Random Split", color="steelblue", alpha=0.85)
bars2 = ax.barh([y + bar_height/2 for y in y_pos], pivot["Temporal"], bar_height,
                label="Temporal Split", color="coral", alpha=0.85)

ax.set_yticks(list(y_pos))
ax.set_yticklabels([f"{m} ({f})" for m, f in pivot.index])
ax.set_xlabel("Accuracy")
ax.set_title("Random vs Temporal Split — Accuracy Comparison")
ax.legend(loc="lower right")

for bars in [bars1, bars2]:
    for bar in bars:
        width = bar.get_width()
        ax.text(width + 0.002, bar.get_y() + bar.get_height() / 2,
                f"{width:.4f}", va="center", fontsize=9)

plt.tight_layout()
plt.show()

# Show the accuracy drop
print("\nAccuracy drop (Random → Temporal):")
for (model, feat), row in pivot.iterrows():
    drop = row["Random"] - row["Temporal"]
    print(f"  {model} ({feat}): {row['Random']:.4f} → {row['Temporal']:.4f}  (Δ = {drop:+.4f})")

## 6. Confusion Matrices — Best Model per Split

Compare the best model's confusion matrix on random vs temporal test sets.

In [ ]:
# Best model per split
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, split_name, data in zip(axes, ["Random", "Temporal"], [random_data, temporal_data]):
    # Find best result for this split
    split_results = {k: v for k, v in all_results.items() if v["split"] == split_name}
    best_name, best_res = max(split_results.items(), key=lambda x: x[1]["accuracy"])

    cm = confusion_matrix(data["y_test"], best_res["y_pred"])
    disp = ConfusionMatrixDisplay(cm, display_labels=best_res["le"].classes_)
    disp.plot(ax=ax, cmap="Blues", values_format="d")
    ax.set_title(f"{split_name} Split — {best_name.split(' (')[0]}\n(acc={best_res['accuracy']:.4f})")

plt.tight_layout()
plt.show()

## 7. ROC Curves (One-vs-Rest) — Random vs Temporal

Plot per-class ROC curves for the best model from each split side by side.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, split_name, data in zip(axes, ["Random", "Temporal"], [random_data, temporal_data]):
    # Best model for this split (filtered features for probability support)
    split_filt = {k: v for k, v in all_results.items()
                  if v["split"] == split_name and v["feature_set"] == "Filtered"}
    best_name, best_res = max(split_filt.items(), key=lambda x: x[1]["accuracy"])
    best_model = best_res["model"]
    le = best_res["le"]

    X_test_best = data["X_test_filt"]

    if hasattr(best_model, "predict_proba"):
        y_score = best_model.predict_proba(X_test_best)
    else:
        y_score = best_model.decision_function(X_test_best)

    n_classes = len(le.classes_)
    y_test_bin = label_binarize(data["y_test"], classes=range(n_classes))

    for i, class_name in enumerate(le.classes_):
        fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_score[:, i])
        roc_auc = auc(fpr, tpr)
        ax.plot(fpr, tpr, label=f"{class_name} (AUC={roc_auc:.3f})")

    ax.plot([0, 1], [0, 1], "k--", alpha=0.3)
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.set_title(f"{split_name} Split — {best_name.split(' (')[0]}")
    ax.legend(loc="lower right", fontsize=9)

plt.tight_layout()
plt.show()

## 8. Feature Importance — Random vs Temporal

Compare which features matter most when the model is trained on historical data vs random data.

In [ ]:
# Best tree-based model (Filtered features) from each split
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

for ax, split_name, data in zip(axes, ["Random", "Temporal"], [random_data, temporal_data]):
    # Find best tree-based filtered model for this split
    tree_results = {k: v for k, v in all_results.items()
                    if v["split"] == split_name and v["feature_set"] == "Filtered"
                    and hasattr(v["model"], "feature_importances_")}

    if not tree_results:
        ax.text(0.5, 0.5, "No tree models", ha="center", va="center")
        continue

    best_name, best_res = max(tree_results.items(), key=lambda x: x[1]["accuracy"])
    importances = pd.Series(
        best_res["model"].feature_importances_, index=data["X_train_filt"].columns
    )
    top20 = importances.nlargest(20)
    top20.sort_values().plot.barh(ax=ax, color="steelblue")
    ax.set_title(f"{split_name} Split — {best_name.split(' (')[0]}\nTop 20 Features")
    ax.set_xlabel("Importance (Gini)")

plt.tight_layout()
plt.show()

## 9. Save Best Models

In [ ]:
# Save best model from each split
for split_name in ["Random", "Temporal"]:
    split_results = {k: v for k, v in all_results.items() if v["split"] == split_name}
    best_name, best_res = max(split_results.items(), key=lambda x: x[1]["accuracy"])
    out_dir = f"../data/processed/{split_name.lower()}"
    joblib.dump(best_res["model"], f"{out_dir}/best_model.pkl")
    print(f"{split_name}: {best_name.split(' (')[0]} (acc={best_res['accuracy']:.4f}) → {out_dir}/best_model.pkl")

# Save full comparison table
summary.to_csv("../data/processed/model_comparison.csv", index=False)
print(f"\nComparison table → ../data/processed/model_comparison.csv")